In [1]:
# Step 1: Import Libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 2: Sample Text Corpus (you can replace with bigger dataset)
corpus = [
    "I love deep learning",
    "I love natural language processing",
    "I enjoy learning new things",
    "deep learning models are powerful",
    "language models can predict words",
]

In [2]:
# Step 3: Tokenize Text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index)+1
print("Vocabulary Size:", total_words)

Vocabulary Size: 17


In [3]:
# Step 4: Create Input Sequences for Next Word Prediction
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

# Pad sequences to equal length
max_seq_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre"))


In [4]:
X = input_sequences[:, :-1]  # features
y = input_sequences[:, -1]   # labels (next word)
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Input Shape:", X.shape)
print("Output Shape:", y.shape)


Input Shape: (19, 4)
Output Shape: (19, 17)


In [5]:
# Step 5: Build Bidirectional lstm Model
model = Sequential()
model.add(Embedding(total_words, 64, input_length=max_seq_len-1))
model.add(Bidirectional(LSTM(64)))
model.add(Dense(total_words, activation="softmax"))

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

f:\QSP\Gen AI\deeplearning\DELENV\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
print("Total words:", total_words)
print("y shape:", y.shape)


Total words: 17
y shape: (19, 17)


In [7]:
# Step 6: Train Model
history = model.fit(X, y, epochs=200, verbose=1)


Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - accuracy: 0.0526 - loss: 2.8301
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 684ms/step - accuracy: 0.1579 - loss: 2.8236
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.1579 - loss: 2.8171
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - accuracy: 0.1579 - loss: 2.8104
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 665ms/step - accuracy: 0.1579 - loss: 2.8035
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.2105 - loss: 2.7964
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.2105 - loss: 2.7889
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - accuracy: 0.2105 - loss: 2.7811
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.2105 - loss: 2.7727
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.2105 - loss: 2.7637
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step - accuracy: 0.2105 - loss: 2.7541
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.

In [8]:
# Step 7: Predict Next Word Function
def predict_next_word(seed_text, next_words=1):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding="pre")
        predicted = np.argmax(model.predict(token_list), axis=-1)[0]
        
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

In [9]:
# Step 8: Test the Model
print(predict_next_word("i love deep ",1))
print(predict_next_word("I enjoy", 3))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 693ms/step
i love deep  learning
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
I enjoy learning new things
